In [1]:
!pip install -q transformers==4.52.4 gradio pydantic pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 74.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 59.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 68.5 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [3]:
def call_llm(prompt, max_new_tokens=500):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True).strip()

print(call_llm("Say hello in one sentence."))

Hello!


In [4]:
import json, re
from pydantic import BaseModel

class Task(BaseModel):
    task: str
    owner: str
    deadline: str

class MeetingReport(BaseModel):
    summary: str
    decisions: list[str]
    tasks: list[Task]

In [5]:
def summarize(transcript):
    return call_llm(
        "Write a short summary of this meeting as one paragraph of plain prose, "
        "at most 3 sentences. Do not use bullet points or lists. "
        "Describe the main topics and outcomes only, and do not mention "
        "individual deadlines.\n\n"
        f"Transcript:\n{transcript}\n\nSummary:",
        max_new_tokens=200
    )
def extract_decisions(transcript):
    return call_llm(
        "List only the decisions made in this meeting, one per line, "
        "each line starting with '-'. Do not include action items or tasks. "
        "Return only the list.\n\n"
        f"Transcript:\n{transcript}\n\nDecisions:",
        max_new_tokens=200
    )

def extract_tasks(transcript):
    return call_llm(
        "Extract the action items from this meeting as a JSON list. "
        'Each item must be: {"task": "...", "owner": "...", "deadline": "..."}. '
        'Only use names that appear in the transcript. '
        'Use "Not specified" if the owner or deadline is missing. '
        "Return ONLY the JSON list, no other text.\n\n"
        f"Transcript:\n{transcript}\n\nJSON:",
        max_new_tokens=400
    )

In [6]:
def parse_tasks(raw):
    match = re.search(r"\[.*\]", raw, re.DOTALL)
    if not match:
        return []
    try:
        return [Task(**t) for t in json.loads(match.group())]
    except Exception:
        return []

def analyze_meeting(transcript, retries=3):
    summary = summarize(transcript)

    decisions_raw = extract_decisions(transcript)
    decisions = [d.strip("-•* ").strip()
                 for d in decisions_raw.split("\n") if d.strip()]

    tasks = []
    for _ in range(retries):
        tasks = parse_tasks(extract_tasks(transcript))
        if tasks:
            break

    return MeetingReport(summary=summary, decisions=decisions, tasks=tasks)

In [7]:
transcript = """Sara: Let's start with the website launch. Are we on track?
Ahmed: Mostly. The design is done, but the payment page still has bugs.
Sara: Okay. We decided to postpone the launch to October 20th.
Ahmed: I'll fix the payment page by next Wednesday.
Mona: I'll prepare the marketing emails by October 15th.
Sara: Also, we agreed to use Stripe instead of PayPal.
Sara: Mona, please send me the budget report by Friday."""

report = analyze_meeting(transcript)
print(report.model_dump_json(indent=2))

{
  "summary": "At the meeting, the team discussed the website launch, which was rescheduled to October 20th due to ongoing issues with the payment page. Ahmed committed to fixing these bugs by the following Wednesday, while Mona agreed to prepare marketing emails by October 15th. Additionally, the team decided to switch to Stripe for payment processing, and Mona was tasked with submitting the budget report by the end of the week.",
  "decisions": [
    "Postpone website launch to October 20th",
    "Use Stripe instead of PayPal for payments"
  ],
  "tasks": [
    {
      "task": "Fix payment page bugs",
      "owner": "Ahmed",
      "deadline": "Next Wednesday"
    },
    {
      "task": "Prepare marketing emails",
      "owner": "Mona",
      "deadline": "October 15th"
    },
    {
      "task": "Send budget report",
      "owner": "Mona",
      "deadline": "Friday"
    }
  ]
}


In [8]:
import gradio as gr
import pandas as pd

def run(transcript):
    r = analyze_meeting(transcript)
    decisions = "\n".join(f"• {d}" for d in r.decisions)
    table = pd.DataFrame([t.model_dump() for t in r.tasks],
                         columns=["task", "owner", "deadline"])
    return r.summary, decisions, table

demo = gr.Interface(
    fn=run,
    inputs=gr.Textbox(lines=12, label="Meeting transcript"),
    outputs=[gr.Textbox(label="Meeting Summary"),
             gr.Textbox(label="Decisions"),
             gr.Dataframe(label="Tasks (Owner / Deadline)")],
    title="AI Meeting Intelligence",
)
demo.launch(share=True, prevent_thread_lock=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://dec56333f1c0012bd2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.13/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.13/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.13/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
